# What's the best way to shuffle cards?

The purpose of this notebook is to explore and evaluate the different ways of shuffling cards. The goal is therefore to ensure that a deck of cards is "random" after a minimum number of shuffles, meaning that a player should not be able to predict with certainty the result of the next draw. 

To do this, we use Shannon entropy, which measures the uncertainty associated with an event. Thus, the higher the Shannon entropy, the more information will be needed to accurately determine the next

### 1) The objects card and deck

First and foremost, let's define the objects of the study

In [ ]:
from dataclasses import dataclass
from enum import Enum

# Special class for the suit of the card
class Suit(Enum):
    HEARTS = "♥"
    DIAMONDS = "♦"
    CLUBS = "♣"
    SPADES = "♠"

# Frozen = True : impossible to modify a card once it is created
# Order= True : give an automatic comparaison methode
@dataclass(frozen=True, order=True)
class Card:

    # attiribute of the card
    value: int  # 1=Ace, 2-10, 11=Jack, 12=Queen, 13=King
    suit: Suit

    # When definied, the value most be in the range [1,13]
    def __post_init__(self):
        if not 1 <= self.value <= 13:
            raise ValueError(f"Valeur invalide : {self.value} (doit être entre 1 et 13)")

    def __repr__(self):
        names = {1: "A", 11: "J", 12: "Q", 13: "K"}
        label = names.get(self.value, str(self.value))
        return f"{label}{self.suit.value}"

In [ ]:
c1 = Card(1, Suit.SPADES)
c2 = Card(13, Suit.HEARTS)
print(c1, c2)
print(c1 < c2)

A♠ K♥
True


In [5]:
from dataclasses import dataclass, field

# Class Deck
@dataclass
class Deck:
    cards: list[Card] = field(default_factory=list)

    # Initialisation of the deck during the creation of the instance
    def __post_init__(self):
        if not self.cards:
            self.cards = self._new_deck_order()

    # Method to initialise the deck as Aces, 2, 3, 4 ... J, Q, K
    @staticmethod
    def _new_deck_order() -> list[Card]:
        """Ordre standard d'un paquet neuf : par couleur, As à Roi."""
        suit_order = [Suit.CLUBS, Suit.DIAMONDS, Suit.HEARTS, Suit.SPADES]
        return [Card(value, suit) for suit in suit_order for value in range(1, 14)]

    # len = 52
    def __len__(self) -> int:
        return len(self.cards)

    # deck[i] return the i-th card
    def __getitem__(self, index):
        return self.cards[index]

    # If we want to us a for loop
    def __iter__(self):
        return iter(self.cards)

    def __repr__(self) -> str:
        return f"Deck({len(self.cards)} cards, top card: {self.cards[0] if self.cards else None})"

    def reset(self) -> None:
        """Reset the deck to its initial order"""
        self.cards = self._new_deck_order()

    def copy(self) -> "Deck":
        """Return an independant copy of the deck"""
        return Deck(cards=self.cards.copy())

In [7]:
d = Deck()
print(d)
print(d[0], d[-1])  # first card, last card

Deck(52 cards, top card: A♣)
A♣ K♠


## 2) Shannon entropie for the position of a card

For each card $c$, after shuffling $N$ times (starting from the same sorted deck), we examine the empirical distribution of its final position: $p_c(j) =$ the probability that card $c$ ends up in position $j$. If the shuffling has almost no effect, $p_c$​ is concentrated around its original position (low entropy, close to $0$).
If the shuffling is perfectly random (ideal Fisher-Yates), $p_c$​ becomes uniformly distributed across the $52$ positions (maximum entropy $= \log_2(52) ≈ 5.7$ bits per card).

In [39]:
import numpy as np
from collections import Counter


def position_entropy(decks: list[Deck], reference: Deck) -> float:
    """
    Compute the mean entropy per card, over the entire shuffled deck, 
    compared to the non-shuffled deck.

    Return the mean entropy in bits (0 = aucun mélange, log2(n) = perfectly random).

    decks: list of deck to evaluate shuffled the same way
    reference: the non-shuffled deck
    """

    # Number of cards
    n = len(reference)
    # Number of decks to evaluate
    N = len(decks)

    # Initial position of each card
    ref_position = {card: i for i, card in enumerate(reference.cards)}

    # For each card, we cumulate the empirical distribution of its final position
    # Counter allow to keep in memory the occurance of the card position 
    # (as a dictionnary key: position, value: number of time it was observed)
    position_counts = {card: Counter() for card in reference.cards}

    # For each deck, we evaluate the position of the card
    for deck in decks:
        for position, card in enumerate(deck.cards):
            # +1 to the counter of position for this card
            position_counts[card][position] += 1

    entropies = []
    # Conversion of the position to a probability of event
    # For each card of the deck (count = dictionnary the count the position)
    for card, counts in position_counts.items():
        # counts.values() = number of time that this position was observed (at least 1 time)
        # / N , to transform it into a probability
        probs = np.array(list(counts.values())) / N
        # h(p) = - p * ln(p)
        h = -np.sum(probs * np.log2(probs))
        # Then the entropy for this card is added
        entropies.append(h)

    # And we return the mean entropy on all the card
    return float(np.mean(entropies))

In [40]:
# Reference deck
reference = Deck()

# Shuffled deck (the same for the moment)
test = reference.copy()

# Creation of a list of deck shuffled the same way
shuffled_decks = [test for _ in range(20)]

# Evaluation of the entropy
h = position_entropy(shuffled_decks, reference)

# For the moment, H = 0 because it is the same deck
print(f"Mean Entropy : {h:.2f} bits (theorical max : {np.log2(52):.2f})")

Mean Entropy : 0.00 bits (theorical max : 5.70)
